# 🚒 CFR EVO: Fine-Tune Whisper on Coquitlam Fire Rescue Dispatches

This Google Colab notebook fine-tunes `openai/whisper-base` using **LoRA (Low-Rank Adaptation)** on your verified dispatch recordings.
Once trained, it exports the model into `ctranslate2` format, ready to run locally on your Raspberry Pi / Kiosk with `faster-whisper`!

--- 
### 📋 Instructions:
1. Click **Runtime** -> **Change runtime type** -> Select **T4 GPU**.
2. Upload your `cfr_whisper_dataset.zip` file to the Colab files sidebar on the left.
3. Run all cells below sequentially!

In [ ]:
# 1. Install Dependencies
!pip install --quiet --upgrade pip
!pip install --quiet transformers datasets peft accelerate evaluate jiwer soundfile librosa ctranslate2

In [ ]:
# 2. Unzip Dataset and Load into Hugging Face Dataset
import os
import zipfile
import pandas as pd
from datasets import Dataset, Audio

if os.path.exists('cfr_whisper_dataset.zip'):
    with zipfile.ZipFile('cfr_whisper_dataset.zip', 'r') as zip_ref:
        zip_ref.extractall('dataset')
    print('✅ Dataset unzipped successfully!')
else:
    print('⚠️ Please upload cfr_whisper_dataset.zip to Colab files sidebar first!')

df = pd.read_csv('dataset/metadata.csv')
df['audio'] = 'dataset/audio/' + df['file_name']
df['sentence'] = df['verified_transcript']

dataset = Dataset.from_pandas(df[['audio', 'sentence']])
dataset = dataset.cast_column('audio', Audio(sampling_rate=16000))
print(f'✅ Dataset loaded: {len(dataset)} samples ready!')

In [ ]:
# 3. Load Model Components & Data Collator
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from transformers import WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor, WhisperForConditionalGeneration

model_id = 'openai/whisper-base'
feature_extractor = WhisperFeatureExtractor.from_pretrained(model_id)
tokenizer = WhisperTokenizer.from_pretrained(model_id, language='English', task='transcribe')
processor = WhisperProcessor.from_pretrained(model_id, language='English', task='transcribe')

def prepare_dataset(batch):
    audio = batch['audio']
    batch['input_features'] = feature_extractor(audio['array'], sampling_rate=audio['sampling_rate']).input_features[0]
    batch['labels'] = tokenizer(batch['sentence']).input_ids
    return batch

dataset = dataset.map(prepare_dataset, remove_columns=dataset.column_names)

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{'input_features': feature['input_features']} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors='pt')
        label_features = [{'input_ids': feature['labels']} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors='pt')
        labels = labels_batch['input_ids'].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch['labels'] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
print('✅ Preprocessing & collator initialized!')

In [ ]:
# 4. Apply LoRA (PEFT) Adapter to Whisper Base
from peft import LoraConfig, get_peft_model, LoraModel

model = WhisperForConditionalGeneration.from_pretrained(model_id)
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

config = LoraConfig(r=32, lora_alpha=64, target_modules=['q_proj', 'v_proj'], lora_dropout=0.05, bias='none')
model = get_peft_model(model, config)
model.print_trainable_parameters()
print('✅ LoRA config attached!')

In [ ]:
# 5. Train Model using Seq2SeqTrainer
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir='./whisper-base-cfr-output',
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-3,
    warmup_steps=10,
    num_train_epochs=5,
    evaluation_strategy='no',
    fp16=True,
    save_strategy='epoch',
    logging_steps=5,
    report_to='none'
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=dataset,
    data_collator=data_collator,
    tokenizer=processor.feature_extractor,
)

print('🔥 Starting LoRA Fine-Tuning Training...')
trainer.train()
print('🎉 Training Complete!')

In [ ]:
# 6. Merge & Convert Model to CTranslate2 Format for faster-whisper
import ctranslate2

# Merge LoRA weights into base model
merged_model = model.merge_and_unload()
merged_model.save_pretrained('./merged_whisper_base')
processor.save_pretrained('./merged_whisper_base')

# Convert to CTranslate2 format (int8 quantized for ultrafast Pi/Kiosk execution)
!ct2-transformers-converter --model ./merged_whisper_base --output_dir ./whisper-base-cfr-ct2 --quantization int8

# Zip final converted model for download
!zip -r whisper-base-cfr-ct2.zip ./whisper-base-cfr-ct2
print('🚀 Fine-tuned CTranslate2 model zipped: whisper-base-cfr-ct2.zip ready to download!')